In [ ]:
library(plyr)
library(dplyr)
library(tidyr)
library(lme4)
library(lmerTest)
library(interactions)
library(ggplot2)
library(emmeans)
library(rempsyc)
library(patchwork)

# TF measures predicted by acc * soc * age

## Load data

In [ ]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- "/Users/fzaki001/thrive-theta-ddm/results/figures/ms"
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/", analysis_path)

id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'power_early',
    'ITPS_early',
    'ICPS_early_DLPFC_collapsed',
    'ICPS_early_MOTOR_collapsed',
    'ICPS_early_OCC_collapsed',
    'bfne_b_scrdTotal_s1_r1_e1'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))

tf_data$soc <- as.factor(tf_data$soc)
tf_data$sub <- as.factor(tf_data$sub)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))


# outcome <- "Reaction"
# predictors <- "acc * age_m * soc + sex + dp_inperson + (1 | sub)"

# # Create the string formula
# full_formula_string <- paste(outcome, "~", predictors)
# formula <- as.formula(full_formula_string)
# Convert and run
# model <- lmer(as.formula(full_formula_string), data = sleepstudy)

## Power

In [ ]:
label <- "power_bfne"
model <- lmer(power_early ~ acc * age_m * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)
lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

In [ ]:
interact_plot(model,
              pred = "acc",
              modx="bfne_b_scrdTotal_s1_r1_e1",
              interval = 1,
              dodge.width = 0.2,
              y.label = "Power",
              x.label = "BFNE",
             ) + theme(legend.position = "bottom")

## ITPS

In [ ]:
label <- "ITPS_bfne"
model <- lmer(ITPS_early ~ acc * age_m * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)
lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

In [ ]:
interact_plot(model,
              pred = "acc",
              modx="bfne_b_scrdTotal_s1_r1_e1",
              mod2="soc",
              interval = 1,
              dodge.width = 0.2,
              y.label = "ITPS",
              # x.label = "BFNE",
             ) + theme(legend.position = "bottom")

In [ ]:
# 1. Define the variables for clarity
# Replace "Level_A" and "Level_B" with your actual acc1 factor levels (e.g., 0 and 1)
# If acc1 is numeric, choose specific values like mean +/- SD
my_labels <- c("Low age", "Mean age", "High age")
# --- Plot for Acc Level 1 ---
p1 <- interact_plot(
  model,
  pred = bfne_b_scrdTotal_s1_r1_e1,
  modx = soc,
  mod2 = age_m,
    mod2.labels = my_labels,
  at = list(acc = "Error"),  # <--- ISOLATE ACC LEVEL HERE
  # plot.points = TRUE            # Optional: show raw data
) +
  ggtitle("Error") +
  theme_minimal()

# --- Plot for Acc Level 2 ---
p2 <- interact_plot(
  model,
  pred = bfne_b_scrdTotal_s1_r1_e1,
  modx = soc,
  mod2 = age_m,
    mod2.labels = my_labels,
  at = list(acc = "Correct"),  # <--- ISOLATE ACC LEVEL HERE
  # plot.points = TRUE
) +
  ggtitle("Correct") +
  theme_minimal()

# --- Combine them ---
# This creates a side-by-side comparison
# p1 + p2

combined_plot <- p1 + p2 + 
  plot_layout(guides = "collect") # Optional: merges legends to save even more space

# Save with explicit width (in inches)
ggsave(sprintf("wide_interaction_plot_%s.png", label), combined_plot, width = 16, height = 6, dpi = 300)

## MFC-DLPFC ICPS

In [ ]:
label <- "ICPS_DLPFC_bfne"
model <- lmer(ICPS_early_DLPFC_collapsed ~ acc * age_m * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)
lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

In [ ]:
# 1. Define the variables for clarity
# Replace "Level_A" and "Level_B" with your actual acc1 factor levels (e.g., 0 and 1)
# If acc1 is numeric, choose specific values like mean +/- SD
my_labels <- c("Low age", "Mean age", "High age")
# --- Plot for Acc Level 1 ---
p1 <- interact_plot(
  model,
  pred = bfne_b_scrdTotal_s1_r1_e1,
  modx = soc,
  mod2 = age_m,
    mod2.labels = my_labels,
  at = list(acc = "Error"),  # <--- ISOLATE ACC LEVEL HERE
  # plot.points = TRUE            # Optional: show raw data
) +
  ggtitle("Error") +
  theme_minimal()

# --- Plot for Acc Level 2 ---
p2 <- interact_plot(
  model,
  pred = bfne_b_scrdTotal_s1_r1_e1,
  modx = soc,
  mod2 = age_m,
    mod2.labels = my_labels,
  at = list(acc = "Correct"),  # <--- ISOLATE ACC LEVEL HERE
  # plot.points = TRUE
) +
  ggtitle("Correct") +
  theme_minimal()

# --- Combine them ---
# This creates a side-by-side comparison
# p1 + p2

combined_plot <- p1 + p2 + 
  plot_layout(guides = "collect") # Optional: merges legends to save even more space

# Save with explicit width (in inches)
ggsave(sprintf("wide_interaction_plot_%s.png", label), combined_plot, width = 16, height = 6, dpi = 300)

In [ ]:
interact_plot(model,
              pred = "age_m",
              modx="bfne_b_scrdTotal_s1_r1_e1",
              mod2="soc",
              interval = 1,
              dodge.width = 0.2,
              y.label = "ICPS DLPFC",
              # x.label = "BFNE",
             ) + theme(legend.position = "bottom")

## MFC-MOTOR ICPS

In [ ]:
label <- "ICPS_MOTOR_bfne"
model <- lmer(ICPS_early_MOTOR_collapsed ~ acc * age_m * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)
lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

## MFC-OCC ICPS

In [ ]:
label <- "ICPS_OCC_bfne"
model <- lmer(ICPS_early_OCC_collapsed ~ acc * age_m * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)
lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

In [ ]:
interact_plot(model,
              pred = "age_m",
              modx="bfne_b_scrdTotal_s1_r1_e1",
              mod2="soc",
              interval = 1,
              dodge.width = 0.2,
              y.label = "ICPS OCC",
              # x.label = "BFNE",
             ) + theme(legend.position = "bottom")

# DDM measures predicted by ICPS

## Load data

In [ ]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- "/Users/fzaki001/thrive-theta-ddm/results/figures/ms"
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/", analysis_path)

# it doesnt matter 0 or 1 because the models below use difference scores; because of that we need to get rid of duplicates (diff scores are the same for acc=0 and acc=1)
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'reversed_ratio_diff',
    'a_diff',
    'p_diff',
    'ter_diff',
    'ICPS_early_DLPFC_diff_collapsed',
    'ICPS_early_MOTOR_diff_collapsed',
    'ICPS_early_OCC_diff_collapsed',
    'bfne_b_scrdTotal_s1_r1_e1'
    
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))
tf_data <- subset(tf_data, tf_data$acc == 1)

tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))

# contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

## sda/rd (attentional ratio)

In [ ]:
label <- "ratio_reversed"
plot_label <- bquote(paste('Attentional Control', 'sd'['a'], '/r'['d'], '(Post-error - Post-correct)'))
model <- lmer(reversed_ratio_diff ~ ICPS_early_OCC_diff_collapsed * age_m  * soc* bfne_b_scrdTotal_s1_r1_e1 + ICPS_early_DLPFC_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

In [ ]:
label <- "ratio_reversed_motor"
plot_label <- bquote(paste('Attentional Control', 'sd'['a'], '/r'['d'], '(Post-error - Post-correct)'))
model <- lmer(reversed_ratio_diff ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc* bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

## a (boundary separation)

In [ ]:
interact_plot(model,
              pred = "age_m",
              modx="bfne_b_scrdTotal_s1_r1_e1",
              interval = 1,
              dodge.width = 0.2,
              y.label = "SSP DDM ratio (post-error - post-correct)",
              # x.label = "BFNE",
             ) + theme(legend.position = "bottom")

In [ ]:
label <- "boundary_separation_bfne"
plot_label <- "boundary separation (Error - Correct)"
model <- lmer(a_diff ~ ICPS_early_OCC_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + ICPS_early_DLPFC_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

In [ ]:
interact_plot(model,
              pred = "ICPS_early_DLPFC_diff_collapsed",
              modx="bfne_b_scrdTotal_s1_r1_e1",
              interval = 1,
              dodge.width = 0.2,
              y.label = "SSP DDM boundary (post-error - post-correct)",
              # x.label = "BFNE",
             ) + theme(legend.position = "bottom")

In [ ]:
interact_plot(model,
              pred = "ICPS_early_OCC_diff_collapsed",
              modx="bfne_b_scrdTotal_s1_r1_e1",
              mod2="soc",
              interval = 1,
              dodge.width = 0.2,
              y.label = "SSP DDM boundary (post-error - post-correct)",
              # x.label = "BFNE",
             ) + theme(legend.position = "bottom")

In [ ]:
label <- "boundary_separation_motor_bfne"
plot_label <- "boundary separation (Error - Correct)"
model <- lmer(a_diff ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

# Behavioral post-error adjustments measures predicted by ICPS

## Load data

In [ ]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- "/Users/fzaki001/thrive-theta-ddm/results/figures/ms"
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/", analysis_path)

# it doesnt matter 0 or 1 because the models below use difference scores; because of that we need to get rid of duplicates (diff scores are the same for acc=0 and acc=1)
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'pea',
   'peri_rt',
   'pes',
   'ICPS_early_DLPFC_diff_collapsed',
   'ICPS_early_MOTOR_diff_collapsed',
   'ICPS_early_OCC_diff_collapsed',
   'bfne_b_scrdTotal_s1_r1_e1'
  )
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))
tf_data <- subset(tf_data, tf_data$acc == 1)

tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
# tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

# contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

## PEA

In [ ]:
label <- "PEA_bfne"
model <- lmer(pea ~ ICPS_early_OCC_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + ICPS_early_DLPFC_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

In [ ]:
label <- "PEA_motor_bfne"
model <- lmer(pea ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

## PERI

In [ ]:
label <- "PERI_bfne"
model <- lmer(peri_rt ~ ICPS_early_OCC_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + ICPS_early_DLPFC_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

In [ ]:
label <- "PERI_motor_bfne"
model <- lmer(peri_rt ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

## PES

In [ ]:
label <- "PES_bfne"
model <- lmer(pes ~ ICPS_early_OCC_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + ICPS_early_DLPFC_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

In [ ]:
label <- "PES_motor_bfne"
plot_y <- "PES"
model <- lmer(pes ~ ICPS_early_MOTOR_diff_collapsed * age_m  * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub), 
              data = tf_data)
summary(model)

lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

In [ ]:
# test(emtrends(model, pairwise ~ age_m | soc, var="age_m"), adjust="fdr")

# png(file=sprintf("%s/%s.png", pic_path, "perceptual strength"), width=4, height=3, units="in", res=600)
interact_plot(model, pred = "age_m", modx="soc", interval = 1, dodge.width = 0.2, y.label = plot_y) + theme(legend.position = "bottom")

# ERN predicted by acc * soc * age

## Load data

In [ ]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- "/Users/fzaki001/thrive-theta-ddm/results/figures/ms"
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/", analysis_path)

id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'ERN',
    'ERN_laplacian',
    'bfne_b_scrdTotal_s1_r1_e1'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))

tf_data$soc <- as.factor(tf_data$soc)
tf_data$sub <- as.factor(tf_data$sub)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

In [ ]:
label <- "ERN_bfne"
model <- lmer(ERN ~ acc * age_m * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)
lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))

In [ ]:
label <- "ERN_laplacian_bfne"
model <- lmer(ERN_laplacian ~ acc * age_m * soc * bfne_b_scrdTotal_s1_r1_e1 + sex + dp_inperson + (1 | sub),
              data = tf_data)
summary(model)
lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label))